In [ ]:
import requests
import json

# =========================
# CONFIG
# =========================
OPENROUTER_API_KEY = "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"
MODEL = "arcee-ai/trinity-large-preview:free"


OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

# =========================
# SENTENCE → SQL
# =========================
def sentence_to_sql(sentence: str) -> str:
    prompt = f"""
You are an expert MySQL query generator with intelligent query classification capabilities.

TASK 1: CLASSIFY THE QUERY TYPE
Analyze the user's question and determine if it requires:
1. DIRECT_SQL - Simple SQL query only
2. SQL_WITH_STATS - SQL + statistical analysis (variance, averages, consistency)
3. PATTERN_DETECTION - Pattern recognition (seasonal, behavioral)
4. ANOMALY_DETECTION - Unusual behavior detection
5. TREND_ANALYSIS - Time-based trends (improving/declining)
6. COMPARISON - Employee vs employee comparison

TASK 2: GENERATE SQL (if applicable)
If the query needs data from database, generate ONLY valid MySQL 8+ SQL.

TASK 3: SPECIFY ADDITIONAL ANALYSIS (if needed)
List what post-SQL processing is required.

OUTPUT FORMAT (JSON):
{{
  "query_type": "DIRECT_SQL|SQL_WITH_STATS|PATTERN_DETECTION|ANOMALY_DETECTION|TREND_ANALYSIS|COMPARISON",
  "sql_query": "SELECT ... (or null if no SQL needed)",
  "analysis_required": ["variance", "trend", "anomaly", "consistency", "pattern"],
  "metric": "attendance|punctuality|work_hours|leaves",
  "time_period": "last_month|last_quarter|last_year",
  "employee_ids": ["emp-001"],
  "visualization": "line_chart|bar_chart|heatmap|pie_chart|null"
}}

STRICT RULES:
- Output ONLY valid JSON
- SQL must be valid MySQL 8+ syntax
- No markdown, no comments, no explanations
- Use employee_id from context if not specified
- For date comparisons, use MySQL date functions

DATABASE SCHEMA:
attendance_records(
  uuid, employee_id, date,
  check_in_time, check_out_time,
  status ENUM('present','absent','late','leave','first_half_leave','second_half_leave','work_from_home'),
  leave_deducted
)
categories(
  uuid, name, description
)
designation(
  uuid, designation, employee_id
)
employees(
  uuid, employee_id, profile_image,
  first_name, last_name,
  official_email, password,
  is_mfa_enabled, contact_no,
  personal_email, pan_number,
  date_of_joining,
  system_role, job_role,
  date_of_relieving
)
employees_skills(
  uuid, employee_id,
  technology_ids, level ENUM('beginner','intermediate','expert','trainee')
)
holidays(
  uuid, name, date, description
)
interns(
  uuid, intern_id, profile_image,
  first_name, last_name,
  personal_email, contact_no,
  role, status ENUM('active','completed'),
  university_name, department,
  date_of_joining, date_of_relieving
)
intern_attendance_records(
  uuid, intern_id, date,
  check_in_time, check_out_time,
  status ENUM('present','absent','late','leave','first_half_leave','second_half_leave','work_from_home'),
  leave_deducted
)
intern_leaves(
  uuid, intern_id,
  request_type ENUM('leave','work_from_home'),
  type ENUM('sick','planned','first_half','second_half'),
  start_date, end_date,
  days, status ENUM('pending','approved','rejected')
)
leaves(
  uuid, employee_id,
  request_type ENUM('leave','work_from_home'),
  type ENUM('sick','planned','first_half','second_half'),
  start_date, end_date,
  days, status ENUM('pending','approved','rejected')
)
projects(
  uuid, employee_id,
  name, market,
  description,
  project_status ENUM('In-Progress','Completed'),
  billing_or_buffer ENUM('billable','buffer')
)
technologies(
  uuid, technology_name,
  category_id
)

CLASSIFICATION EXAMPLES:

Q: "How many days was I present this month?"
A: {{
  "query_type": "DIRECT_SQL",
  "sql_query": "SELECT COUNT(*) as present_days FROM attendance_records WHERE employee_id = 'emp-001' AND status = 'present' AND MONTH(date) = MONTH(CURDATE()) AND YEAR(date) = YEAR(CURDATE())",
  "analysis_required": [],
  "metric": "attendance",
  "time_period": "current_month",
  "employee_ids": ["emp-001"],
  "visualization": null
}}

Q: "How consistent are my work hours?"
A: {{
  "query_type": "SQL_WITH_STATS",
  "sql_query": "SELECT date, TIMESTAMPDIFF(HOUR, check_in_time, check_out_time) as work_hours FROM attendance_records WHERE employee_id = 'emp-001' AND date >= DATE_SUB(CURDATE(), INTERVAL 30 DAY) AND status = 'present'",
  "analysis_required": ["variance", "consistency"],
  "metric": "work_hours",
  "time_period": "last_month",
  "employee_ids": ["emp-001"],
  "visualization": null
}}

Q: "Do I take more leaves during certain months?"
A: {{
  "query_type": "PATTERN_DETECTION",
  "sql_query": "SELECT MONTH(start_date) as month, COUNT(*) as leave_count FROM leaves WHERE employee_id = 'emp-001' AND status = 'approved' GROUP BY MONTH(start_date)",
  "analysis_required": ["seasonal_pattern"],
  "metric": "leaves",
  "time_period": "all_time",
  "employee_ids": ["emp-001"],
  "visualization": "bar_chart"
}}

Q: "Did my attendance show any unusual changes?"
A: {{
  "query_type": "ANOMALY_DETECTION",
  "sql_query": "SELECT date, status, TIMESTAMPDIFF(HOUR, check_in_time, check_out_time) as work_hours FROM attendance_records WHERE employee_id = 'emp-001' AND date >= DATE_SUB(CURDATE(), INTERVAL 90 DAY) ORDER BY date",
  "analysis_required": ["anomaly"],
  "metric": "attendance",
  "time_period": "last_quarter",
  "employee_ids": ["emp-001"],
  "visualization": "line_chart"
}}

Q: "Has my attendance improved over time?"
A: {{
  "query_type": "TREND_ANALYSIS",
  "sql_query": "SELECT DATE_FORMAT(date, '%Y-%m') as month, COUNT(CASE WHEN status = 'present' THEN 1 END) as present_count, COUNT(*) as total_days FROM attendance_records WHERE employee_id = 'emp-001' AND date >= DATE_SUB(CURDATE(), INTERVAL 6 MONTH) GROUP BY DATE_FORMAT(date, '%Y-%m') ORDER BY month",
  "analysis_required": ["trend"],
  "metric": "attendance",
  "time_period": "last_6_months",
  "employee_ids": ["emp-001"],
  "visualization": "line_chart"
}}

Q: "Compare my attendance with John Doe"
A: {{
  "query_type": "COMPARISON",
  "sql_query": "SELECT e.first_name, e.last_name, COUNT(CASE WHEN a.status = 'present' THEN 1 END) as present_days, COUNT(CASE WHEN a.status = 'late' THEN 1 END) as late_days FROM employees e LEFT JOIN attendance_records a ON e.uuid = a.employee_id WHERE e.uuid IN ('emp-001', 'emp-002') AND a.date >= DATE_SUB(CURDATE(), INTERVAL 30 DAY) GROUP BY e.uuid",
  "analysis_required": ["comparison"],
  "metric": "attendance",
  "time_period": "last_month",
  "employee_ids": ["emp-001", "emp-002"],
  "visualization": "bar_chart"
}}

Q: "Show my attendance trend as a graph"
A: {{
  "query_type": "DIRECT_SQL",
  "sql_query": "SELECT date, status FROM attendance_records WHERE employee_id = 'emp-001' AND date >= DATE_SUB(CURDATE(), INTERVAL 30 DAY) ORDER BY date",
  "analysis_required": [],
  "metric": "attendance",
  "time_period": "last_month",
  "employee_ids": ["emp-001"],
  "visualization": "line_chart"
}}

CURRENT USER CONTEXT:
- Employee ID: employee_id
- Current Date: current_date

USER QUESTION:
{sentence}
"""

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": MODEL,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": 0
    }

    response = requests.post(
        OPENROUTER_URL,
        headers=headers,
        json=payload
    )

    response.raise_for_status()

    result = response.json()
    return result["choices"][0]["message"]["content"].strip()


# =========================
# MAIN
# =========================
if __name__ == "__main__":
    print("🧠 Sentence → SQL (Full HRMS Schema)")
    print("----------------------------------")

    sentence = input("Enter your question: ")

    sql = sentence_to_sql(sentence)

    print("\nGenerated SQL Query:")
    print(sql)


🧠 Sentence → SQL (Full HRMS Schema)
----------------------------------

Generated SQL Query:
{
  "query_type": "DIRECT_SQL",
  "sql_query": "SELECT employee_id, first_name, last_name, official_email, date_of_joining FROM employees ORDER BY first_name, last_name",
  "analysis_required": [],
  "metric": null,
  "time_period": null,
  "employee_ids": null,
  "visualization": null
}


In [94]:
import mysql.connector

DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": "kavin@123",
    "database": "dsingz"
}

def execute_sql(sql_query: str):
    conn = mysql.connector.connect(**DB_CONFIG)
    cursor = conn.cursor(dictionary=True)

    cursor.execute(sql_query)
    rows = cursor.fetchall()

    cursor.close()
    conn.close()

    return rows

if __name__ == "__main__":
    rows = execute_sql(sql_query)

    for row in rows:
        print(row)


ProgrammingError: 1064 (42000): You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near '{
  "query_type": "DIRECT_SQL",
  "sql_query": "SELECT employee_id, first_name, ' at line 1

In [95]:
import requests
import json

OPENROUTER_API_KEY = "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"
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL = "deepseek/deepseek-chat"

def sentence_to_sql(sentence: str) -> str:
    prompt = f"""
You are an expert MySQL query generator with intelligent query classification capabilities.

TASK 1: CLASSIFY THE QUERY TYPE
Analyze the user's question and determine if it requires:
1. DIRECT_SQL - Simple SQL query only
2. SQL_WITH_STATS - SQL + statistical analysis (variance, averages, consistency)
3. PATTERN_DETECTION - Pattern recognition (seasonal, behavioral)
4. ANOMALY_DETECTION - Unusual behavior detection
5. TREND_ANALYSIS - Time-based trends (improving/declining)
6. COMPARISON - Employee vs employee comparison

TASK 2: GENERATE SQL (if applicable)
If the query needs data from database, generate ONLY valid MySQL 8+ SQL.

TASK 3: SPECIFY ADDITIONAL ANALYSIS (if needed)
List what post-SQL processing is required.

OUTPUT FORMAT (JSON):
{{
  "query_type": "DIRECT_SQL|SQL_WITH_STATS|PATTERN_DETECTION|ANOMALY_DETECTION|TREND_ANALYSIS|COMPARISON",
  "sql_query": "SELECT ... (or null if no SQL needed)",
  "analysis_required": ["variance", "trend", "anomaly", "consistency", "pattern"],
  "metric": "attendance|punctuality|work_hours|leaves",
  "time_period": "last_month|last_quarter|last_year",
  "employee_ids": ["emp-001"],
  "visualization": "line_chart|bar_chart|heatmap|pie_chart|null"
}}

STRICT RULES:
- Output ONLY valid JSON
- SQL must be valid MySQL 8+ syntax
- No markdown, no comments, no explanations
- Use employee_id from context if not specified
- For date comparisons, use MySQL date functions

DATABASE SCHEMA:
attendance_records(
  uuid, employee_id, date,
  check_in_time, check_out_time,
  status ENUM('present','absent','late','leave','first_half_leave','second_half_leave','work_from_home'),
  leave_deducted
)
categories(
  uuid, name, description
)
designation(
  uuid, designation, employee_id
)
employees(
  uuid, employee_id, profile_image,
  first_name, last_name,
  official_email, password,
  is_mfa_enabled, contact_no,
  personal_email, pan_number,
  date_of_joining,
  system_role, job_role,
  date_of_relieving
)
employees_skills(
  uuid, employee_id,
  technology_ids, level ENUM('beginner','intermediate','expert','trainee')
)
holidays(
  uuid, name, date, description
)
interns(
  uuid, intern_id, profile_image,
  first_name, last_name,
  personal_email, contact_no,
  role, status ENUM('active','completed'),
  university_name, department,
  date_of_joining, date_of_relieving
)
intern_attendance_records(
  uuid, intern_id, date,
  check_in_time, check_out_time,
  status ENUM('present','absent','late','leave','first_half_leave','second_half_leave','work_from_home'),
  leave_deducted
)
intern_leaves(
  uuid, intern_id,
  request_type ENUM('leave','work_from_home'),
  type ENUM('sick','planned','first_half','second_half'),
  start_date, end_date,
  days, status ENUM('pending','approved','rejected')
)
leaves(
  uuid, employee_id,
  request_type ENUM('leave','work_from_home'),
  type ENUM('sick','planned','first_half','second_half'),
  start_date, end_date,
  days, status ENUM('pending','approved','rejected')
)
projects(
  uuid, employee_id,
  name, market,
  description,
  project_status ENUM('In-Progress','Completed'),
  billing_or_buffer ENUM('billable','buffer')
)
technologies(
  uuid, technology_name,
  category_id
)

CLASSIFICATION EXAMPLES:

Q: "How many days was I present this month?"
A: {{
  "query_type": "DIRECT_SQL",
  "sql_query": "SELECT COUNT(*) as present_days FROM attendance_records WHERE employee_id = 'emp-001' AND status = 'present' AND MONTH(date) = MONTH(CURDATE()) AND YEAR(date) = YEAR(CURDATE())",
  "analysis_required": [],
  "metric": "attendance",
  "time_period": "current_month",
  "employee_ids": ["emp-001"],
  "visualization": null
}}

Q: "How consistent are my work hours?"
A: {{
  "query_type": "SQL_WITH_STATS",
  "sql_query": "SELECT date, TIMESTAMPDIFF(HOUR, check_in_time, check_out_time) as work_hours FROM attendance_records WHERE employee_id = 'emp-001' AND date >= DATE_SUB(CURDATE(), INTERVAL 30 DAY) AND status = 'present'",
  "analysis_required": ["variance", "consistency"],
  "metric": "work_hours",
  "time_period": "last_month",
  "employee_ids": ["emp-001"],
  "visualization": null
}}

Q: "Do I take more leaves during certain months?"
A: {{
  "query_type": "PATTERN_DETECTION",
  "sql_query": "SELECT MONTH(start_date) as month, COUNT(*) as leave_count FROM leaves WHERE employee_id = 'emp-001' AND status = 'approved' GROUP BY MONTH(start_date)",
  "analysis_required": ["seasonal_pattern"],
  "metric": "leaves",
  "time_period": "all_time",
  "employee_ids": ["emp-001"],
  "visualization": "bar_chart"
}}

Q: "Did my attendance show any unusual changes?"
A: {{
  "query_type": "ANOMALY_DETECTION",
  "sql_query": "SELECT date, status, TIMESTAMPDIFF(HOUR, check_in_time, check_out_time) as work_hours FROM attendance_records WHERE employee_id = 'emp-001' AND date >= DATE_SUB(CURDATE(), INTERVAL 90 DAY) ORDER BY date",
  "analysis_required": ["anomaly"],
  "metric": "attendance",
  "time_period": "last_quarter",
  "employee_ids": ["emp-001"],
  "visualization": "line_chart"
}}

Q: "Has my attendance improved over time?"
A: {{
  "query_type": "TREND_ANALYSIS",
  "sql_query": "SELECT DATE_FORMAT(date, '%Y-%m') as month, COUNT(CASE WHEN status = 'present' THEN 1 END) as present_count, COUNT(*) as total_days FROM attendance_records WHERE employee_id = 'emp-001' AND date >= DATE_SUB(CURDATE(), INTERVAL 6 MONTH) GROUP BY DATE_FORMAT(date, '%Y-%m') ORDER BY month",
  "analysis_required": ["trend"],
  "metric": "attendance",
  "time_period": "last_6_months",
  "employee_ids": ["emp-001"],
  "visualization": "line_chart"
}}

Q: "Compare my attendance with John Doe"
A: {{
  "query_type": "COMPARISON",
  "sql_query": "SELECT e.first_name, e.last_name, COUNT(CASE WHEN a.status = 'present' THEN 1 END) as present_days, COUNT(CASE WHEN a.status = 'late' THEN 1 END) as late_days FROM employees e LEFT JOIN attendance_records a ON e.uuid = a.employee_id WHERE e.uuid IN ('emp-001', 'emp-002') AND a.date >= DATE_SUB(CURDATE(), INTERVAL 30 DAY) GROUP BY e.uuid",
  "analysis_required": ["comparison"],
  "metric": "attendance",
  "time_period": "last_month",
  "employee_ids": ["emp-001", "emp-002"],
  "visualization": "bar_chart"
}}

Q: "Show my attendance trend as a graph"
A: {{
  "query_type": "DIRECT_SQL",
  "sql_query": "SELECT date, status FROM attendance_records WHERE employee_id = 'emp-001' AND date >= DATE_SUB(CURDATE(), INTERVAL 30 DAY) ORDER BY date",
  "analysis_required": [],
  "metric": "attendance",
  "time_period": "last_month",
  "employee_ids": ["emp-001"],
  "visualization": "line_chart"
}}

CURRENT USER CONTEXT:
- Employee ID: employee_id
- Current Date: current_date

USER QUESTION:
{sentence}
"""

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost",
        "X-Title": "Sentence to SQL"
    }

    payload = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0
    }

    response = requests.post(
        OPENROUTER_URL,
        headers=headers,
        json=payload
    )
    response.raise_for_status()

    raw_output = response.json()["choices"][0]["message"]["content"]
    parsed = json.loads(raw_output)

    return parsed["sql_query"]


In [ ]:
import mysql.connector

DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": "data@123",
    "database": "dsingz"
}

def execute_sql(sql_query: str):
    try:
        conn = mysql.connector.connect(**DB_CONFIG)
        cursor = conn.cursor(dictionary=True)

        cursor.execute(sql_query)
        rows = cursor.fetchall()
        return rows

    except mysql.connector.Error as err:
        print("❌ SQL Error:", err)
        return []

    finally:
        if cursor:
            cursor.close()
        if conn:
            conn.close()


In [98]:
if __name__ == "__main__":
    print("🧠 Natural Language → SQL → MySQL")
    print("--------------------------------")

    sentence = input("Enter your question: ")

    # ✅ SQL comes from LLM output
    sql_query = sentence_to_sql(sentence)

    print("\nGenerated SQL:")
    print(sql_query)

    # ✅ Execute that SQL
    rows = execute_sql(sql_query)

    print("\nQuery Result:")
    if not rows:
        print("No records found.")
    else:
        for row in rows:
            print(row)


🧠 Natural Language → SQL → MySQL
--------------------------------


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [5]:
import requests
import json
import re
import mysql.connector

# =========================
# CONFIG
# =========================
OPENROUTER_API_KEY = "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"   # 🔴 use a fresh key
MODEL = "deepseek/deepseek-chat"
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": "data@123",
    "database": "dsignz"
}

# =========================
# SQL SAFETY VALIDATION
# =========================
def validate_sql(sql: str):
    forbidden = ["delete", "drop", "truncate", "update", "alter"]
    if any(word in sql.lower() for word in forbidden):
        raise ValueError("❌ Dangerous SQL blocked")
    return sql

# =========================
# EXTRACT SQL FROM LLM OUTPUT
# =========================
def extract_sql_from_model_output(raw_output: str) -> str:
    raw_output = raw_output.strip()

    # 1️⃣ Try direct JSON
    try:
        parsed = json.loads(raw_output)
        return parsed["sql_query"]
    except Exception:
        pass

    # 2️⃣ Try extracting JSON block
    match = re.search(r'\{[\s\S]*?\}', raw_output)
    if match:
        try:
            parsed = json.loads(match.group())
            return parsed["sql_query"]
        except Exception:
            pass

    # 3️⃣ Fallback → assume raw SQL
    return raw_output

# =========================
# SENTENCE → SQL (LLM)
# =========================
def sentence_to_sql(sentence: str) -> str:
    prompt = f"""
You are an expert MySQL query generator with intelligent query classification capabilities.

TASK 1: CLASSIFY THE QUERY TYPE
Analyze the user's question and determine if it requires:
1. DIRECT_SQL - Simple SQL query only
2. SQL_WITH_STATS - SQL + statistical analysis (variance, averages, consistency)
3. PATTERN_DETECTION - Pattern recognition (seasonal, behavioral)
4. ANOMALY_DETECTION - Unusual behavior detection
5. TREND_ANALYSIS - Time-based trends (improving/declining)
6. COMPARISON - Employee vs employee comparison

TASK 2: GENERATE SQL (if applicable)
If the query needs data from database, generate ONLY valid MySQL 8+ SQL.

TASK 3: SPECIFY ADDITIONAL ANALYSIS (if needed)
List what post-SQL processing is required.

OUTPUT FORMAT (JSON):
{{
  "query_type": "DIRECT_SQL|SQL_WITH_STATS|PATTERN_DETECTION|ANOMALY_DETECTION|TREND_ANALYSIS|COMPARISON",
  "sql_query": "SELECT ... (or null if no SQL needed)",
  "analysis_required": ["variance", "trend", "anomaly", "consistency", "pattern"],
  "metric": "attendance|punctuality|work_hours|leaves",
  "time_period": "last_month|last_quarter|last_year",
  "employee_ids": ["emp-001"],
  "visualization": "line_chart|bar_chart|heatmap|pie_chart|null"
}}

STRICT RULES:
- Output ONLY valid JSON
- SQL must be valid MySQL 8+ syntax
- No markdown, no comments, no explanations
- Use employee_id from context if not specified
- For date comparisons, use MySQL date functions

DATABASE SCHEMA:
attendance_records(
  uuid, employee_id, date,
  check_in_time, check_out_time,
  status ENUM('present','absent','late','leave','first_half_leave','second_half_leave','work_from_home'),
  leave_deducted
)
categories(
  uuid, name, description
)
designation(
  uuid, designation, employee_id
)
employees(
  uuid, employee_id, profile_image,
  first_name, last_name,
  official_email, password,
  is_mfa_enabled, contact_no,
  personal_email, pan_number,
  date_of_joining,
  system_role, job_role,
  date_of_relieving
)
employees_skills(
  uuid, employee_id,
  technology_ids, level ENUM('beginner','intermediate','expert','trainee')
)
holidays(
  uuid, name, date, description
)
interns(
  uuid, intern_id, profile_image,
  first_name, last_name,
  personal_email, contact_no,
  role, status ENUM('active','completed'),
  university_name, department,
  date_of_joining, date_of_relieving
)
intern_attendance_records(
  uuid, intern_id, date,
  check_in_time, check_out_time,
  status ENUM('present','absent','late','leave','first_half_leave','second_half_leave','work_from_home'),
  leave_deducted
)
intern_leaves(
  uuid, intern_id,
  request_type ENUM('leave','work_from_home'),
  type ENUM('sick','planned','first_half','second_half'),
  start_date, end_date,
  days, status ENUM('pending','approved','rejected')
)
leaves(
  uuid, employee_id,
  request_type ENUM('leave','work_from_home'),
  type ENUM('sick','planned','first_half','second_half'),
  start_date, end_date,
  days, status ENUM('pending','approved','rejected')
)
projects(
  uuid, employee_id,
  name, market,
  description,
  project_status ENUM('In-Progress','Completed'),
  billing_or_buffer ENUM('billable','buffer')
)
technologies(
  uuid, technology_name,
  category_id
)

CLASSIFICATION EXAMPLES:

Q: "How many days was I present this month?"
A: {{
  "query_type": "DIRECT_SQL",
  "sql_query": "SELECT COUNT(*) as present_days FROM attendance_records WHERE employee_id = 'emp-001' AND status = 'present' AND MONTH(date) = MONTH(CURDATE()) AND YEAR(date) = YEAR(CURDATE())",
  "analysis_required": [],
  "metric": "attendance",
  "time_period": "current_month",
  "employee_ids": ["emp-001"],
  "visualization": null
}}

Q: "How consistent are my work hours?"
A: {{
  "query_type": "SQL_WITH_STATS",
  "sql_query": "SELECT date, TIMESTAMPDIFF(HOUR, check_in_time, check_out_time) as work_hours FROM attendance_records WHERE employee_id = 'emp-001' AND date >= DATE_SUB(CURDATE(), INTERVAL 30 DAY) AND status = 'present'",
  "analysis_required": ["variance", "consistency"],
  "metric": "work_hours",
  "time_period": "last_month",
  "employee_ids": ["emp-001"],
  "visualization": null
}}

Q: "Do I take more leaves during certain months?"
A: {{
  "query_type": "PATTERN_DETECTION",
  "sql_query": "SELECT MONTH(start_date) as month, COUNT(*) as leave_count FROM leaves WHERE employee_id = 'emp-001' AND status = 'approved' GROUP BY MONTH(start_date)",
  "analysis_required": ["seasonal_pattern"],
  "metric": "leaves",
  "time_period": "all_time",
  "employee_ids": ["emp-001"],
  "visualization": "bar_chart"
}}

Q: "Did my attendance show any unusual changes?"
A: {{
  "query_type": "ANOMALY_DETECTION",
  "sql_query": "SELECT date, status, TIMESTAMPDIFF(HOUR, check_in_time, check_out_time) as work_hours FROM attendance_records WHERE employee_id = 'emp-001' AND date >= DATE_SUB(CURDATE(), INTERVAL 90 DAY) ORDER BY date",
  "analysis_required": ["anomaly"],
  "metric": "attendance",
  "time_period": "last_quarter",
  "employee_ids": ["emp-001"],
  "visualization": "line_chart"
}}

Q: "Has my attendance improved over time?"
A: {{
  "query_type": "TREND_ANALYSIS",
  "sql_query": "SELECT DATE_FORMAT(date, '%Y-%m') as month, COUNT(CASE WHEN status = 'present' THEN 1 END) as present_count, COUNT(*) as total_days FROM attendance_records WHERE employee_id = 'emp-001' AND date >= DATE_SUB(CURDATE(), INTERVAL 6 MONTH) GROUP BY DATE_FORMAT(date, '%Y-%m') ORDER BY month",
  "analysis_required": ["trend"],
  "metric": "attendance",
  "time_period": "last_6_months",
  "employee_ids": ["emp-001"],
  "visualization": "line_chart"
}}

Q: "Compare my attendance with John Doe"
A: {{
  "query_type": "COMPARISON",
  "sql_query": "SELECT e.first_name, e.last_name, COUNT(CASE WHEN a.status = 'present' THEN 1 END) as present_days, COUNT(CASE WHEN a.status = 'late' THEN 1 END) as late_days FROM employees e LEFT JOIN attendance_records a ON e.uuid = a.employee_id WHERE e.uuid IN ('emp-001', 'emp-002') AND a.date >= DATE_SUB(CURDATE(), INTERVAL 30 DAY) GROUP BY e.uuid",
  "analysis_required": ["comparison"],
  "metric": "attendance",
  "time_period": "last_month",
  "employee_ids": ["emp-001", "emp-002"],
  "visualization": "bar_chart"
}}

Q: "Show my attendance trend as a graph"
A: {{
  "query_type": "DIRECT_SQL",
  "sql_query": "SELECT date, status FROM attendance_records WHERE employee_id = 'emp-001' AND date >= DATE_SUB(CURDATE(), INTERVAL 30 DAY) ORDER BY date",
  "analysis_required": [],
  "metric": "attendance",
  "time_period": "last_month",
  "employee_ids": ["emp-001"],
  "visualization": "line_chart"
}}

CURRENT USER CONTEXT:
- Employee ID: employee_id
- Current Date: current_date

USER QUESTION:
{sentence}
"""

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost",
        "X-Title": "NL to SQL HRMS"
    }

    payload = {
        "model": MODEL,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": 0
    }

    response = requests.post(
        OPENROUTER_URL,
        headers=headers,
        json=payload,
        timeout=30
    )
    response.raise_for_status()

    raw_output = response.json()["choices"][0]["message"]["content"]
    print("\n🔎 RAW MODEL OUTPUT:\n", raw_output)

    sql_query = extract_sql_from_model_output(raw_output)
    return validate_sql(sql_query)

# =========================
# EXECUTE SQL IN MYSQL
# =========================
def execute_sql(sql_query: str):
    try:
        conn = mysql.connector.connect(**DB_CONFIG)
        cursor = conn.cursor(dictionary=True)

        cursor.execute(sql_query)
        rows = cursor.fetchall()
        return rows

    except mysql.connector.Error as err:
        print("❌ MySQL Error:", err)
        return []

    finally:
        if cursor:
            cursor.close()
        if conn:
            conn.close()

# =========================
# MAIN
# =========================
if __name__ == "__main__":
    print("🧠 Natural Language → SQL → MySQL")
    print("--------------------------------")

    sentence = input("Enter your question: ").strip()

    try:
        sql_query = sentence_to_sql(sentence)

        print("\n✅ GENERATED SQL:\n", sql_query)

        rows = execute_sql(sql_query)

        print("\n📊 QUERY RESULT:")
        if not rows:
            print("No records found.")
        else:
            for row in rows:
                print(row)

    except Exception as e:
        print("❌ ERROR:", e)


🧠 Natural Language → SQL → MySQL
--------------------------------

🔎 RAW MODEL OUTPUT:
 ```json
{
  "query_type": "DIRECT_SQL",
  "sql_query": "SELECT employee_id, first_name, last_name, official_email, date_of_joining, job_role FROM employees",
  "analysis_required": [],
  "metric": null,
  "time_period": null,
  "employee_ids": [],
  "visualization": null
}
```

✅ GENERATED SQL:
 SELECT employee_id, first_name, last_name, official_email, date_of_joining, job_role FROM employees

📊 QUERY RESULT:
{'employee_id': 'EMP001', 'first_name': 'Rajesh', 'last_name': 'Kumar', 'official_email': 'rajesh.kumar@company.com', 'date_of_joining': datetime.date(2022, 1, 10), 'job_role': 'desig-001'}
{'employee_id': 'EMP002', 'first_name': 'Anitha', 'last_name': 'Ramesh', 'official_email': 'anitha.ramesh@company.com', 'date_of_joining': datetime.date(2022, 3, 15), 'job_role': 'desig-002'}
{'employee_id': 'EMP003', 'first_name': 'Vijay', 'last_name': 'Krishnan', 'official_email': 'vijay.krishnan@company.